# Notebook 1 — Build RAG From Scratch (No LangChain)
## SRE Runbook Search Assistant

**Goal:** Build a working Retrieval-Augmented Generation pipeline using *only plain Python + numpy + scikit-learn*, so that every step is transparent.

**The system we are building:**

```
5 SRE Runbooks
      │
   Cleaning
      │
   Chunking  (with overlap)
      │
   Embeddings (TF-IDF first — visible & explainable)
      │
   Vector Store (a plain Python list!)
      │
   Similarity Search (cosine similarity, computed by hand)
      │
   Prompt Builder (you will SEE the final prompt)
      │
   LLM Answer (optional — mock answer works without any API key)
```

**Rules of this notebook:**
1. No LangChain, no LlamaIndex, no FAISS, no pgvector. Those come in Notebooks 2–3.
2. Every transformation is visualized **before → after**.
3. Every section ends with an *experiment* — change a number, re-run, observe.

> **Mentor note — why TF-IDF before neural embeddings?**
> A TF-IDF vector has one dimension **per word**, so you can literally read it: "this chunk scores 0.72 on the word `kafka`". A 384-dim neural embedding is a black box. We learn the pipeline on glass, then swap in the black box later — and you'll see that *nothing else changes*.


---
## Section 0 — Setup

Everything here is pre-installed on Google Colab. We only need numpy, pandas, matplotlib, and scikit-learn.


In [ ]:
import re
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine

np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.max_colwidth", 80)
plt.rcParams["figure.figsize"] = (10, 5)

print("Setup complete ✅")

---
## Section 1 — Create the Sample Corpus (5 SRE Runbooks)

Instead of downloading PDFs, we generate the runbooks ourselves. This is important pedagogically:
**you should know your corpus perfectly** so that when retrieval goes wrong later, you can tell *why*.

We deliberately include realistic messiness:
- headers/footers (`CONFIDENTIAL`, page numbers) → motivates **cleaning**
- shell commands that span lines → motivates **chunk overlap**
- overlapping vocabulary between Kafka and the services → motivates **re-ranking** (Notebook 2)


In [ ]:
RAW_RUNBOOKS = {
    "payment-service": """
CONFIDENTIAL - ACME Corp Internal
Page 1 of 2

RUNBOOK: Payment Service Outage

Symptoms:
- 5xx errors on /api/v1/charge endpoint
- Latency above 2000ms on payment-api
- Redis connection timeouts in payment logs

Investigation steps:
1. Check Kubernetes pods for the payment namespace.
   kubectl get pods -n payment
2. Inspect recent logs for exceptions.
   kubectl logs deployment/payment-api -n payment --tail=200
3. Verify Redis connectivity from a payment pod.
   kubectl exec -it payment-api-0 -- redis-cli -h redis-payment ping

Remediation:
If pods are in CrashLoopBackOff, restart the deployment:
   kubectl rollout restart deployment payment-api -n payment
If Redis is unreachable, failover to the replica:
   redis-cli -h redis-payment SENTINEL failover payment-master

Escalation: page the payments on-call if errors persist 15 minutes.

CONFIDENTIAL - ACME Corp Internal
Page 2 of 2
""",

    "order-service": """
CONFIDENTIAL - ACME Corp Internal
Page 1 of 1

RUNBOOK: Order Service Degradation

Symptoms:
- Orders stuck in PENDING state
- Growing queue depth on order-events topic
- Timeouts calling payment-service downstream

Investigation steps:
1. Check order-service pod health.
   kubectl get pods -n orders
2. Check the dead letter queue for poison messages.
   kubectl logs deployment/order-worker -n orders | grep DLQ
3. Verify downstream payment-service is healthy before restarting anything.

Remediation:
Restart the order worker to reprocess stuck orders:
   kubectl rollout restart deployment order-worker -n orders
Replay dead-lettered events after fixing the schema issue.

CONFIDENTIAL - ACME Corp Internal
""",

    "login-service": """
CONFIDENTIAL - ACME Corp Internal

RUNBOOK: Login Service Authentication Failures

Symptoms:
- Spike in 401 responses on /auth/login
- JWT signature validation errors in logs
- Session store (Redis) memory above 90 percent

Investigation steps:
1. Confirm the JWT signing key was not rotated without deployment.
   kubectl get secret jwt-signing-key -n auth -o yaml
2. Check Redis session store memory.
   redis-cli -h redis-sessions INFO memory
3. Review recent deployments to login-service.

Remediation:
Roll back the last deployment if key rotation caused the failure:
   kubectl rollout undo deployment login-service -n auth
Evict expired sessions if Redis memory is exhausted.

Page the identity team if MFA providers are timing out.

CONFIDENTIAL - ACME Corp Internal
""",

    "kafka-cluster": """
CONFIDENTIAL - ACME Corp Internal
Page 1 of 2

RUNBOOK: Kafka Cluster Incidents

Symptoms:
- Consumer lag increasing on critical topics
- Under-replicated partitions alert firing
- ISR shrinking on broker 2
- Producers receiving NotEnoughReplicasException

Investigation steps:
1. Check broker health and disk usage.
   kafka-broker-api-versions.sh --bootstrap-server kafka-0:9092
2. List under-replicated partitions.
   kafka-topics.sh --describe --under-replicated-partitions --bootstrap-server kafka-0:9092
3. Check consumer group lag.
   kafka-consumer-groups.sh --describe --group order-consumers --bootstrap-server kafka-0:9092

Remediation:
If a broker is down due to disk pressure, clear old log segments and restart the broker:
   systemctl restart kafka
If consumer lag keeps growing, scale the consumer group before touching the brokers.
Never delete topics during an incident.

CONFIDENTIAL - ACME Corp Internal
Page 2 of 2
""",

    "kubernetes-deploys": """
CONFIDENTIAL - ACME Corp Internal

RUNBOOK: Kubernetes Deployment Failures

Symptoms:
- Pods stuck in ImagePullBackOff or CrashLoopBackOff
- Rollout stuck at 50 percent
- Readiness probes failing after a new release

Investigation steps:
1. Describe the failing pod to see events.
   kubectl describe pod <pod-name>
2. Check rollout status.
   kubectl rollout status deployment <name>
3. Compare the new image tag against the registry.

Remediation:
Roll back a bad release immediately:
   kubectl rollout undo deployment <name>
Fix readiness probe thresholds if the app needs longer warmup.
Always roll back first, debug second, during a customer-facing incident.

CONFIDENTIAL - ACME Corp Internal
""",
}

print(f"Loaded {len(RAW_RUNBOOKS)} runbooks:")
for name, text in RAW_RUNBOOKS.items():
    print(f"  • {name:20s} {len(text.split()):4d} words, {len(text):5d} chars")

### 1.1 Visualize the corpus

Two quick views: a size comparison, and "document cards" so you internalize what we're indexing.


In [ ]:
names = list(RAW_RUNBOOKS.keys())
word_counts = [len(t.split()) for t in RAW_RUNBOOKS.values()]     # crude length metric: just count
                                                                    # whitespace-separated words per doc
plt.barh(names, word_counts, color="#4C72B0")
plt.xlabel("Word count")
plt.title("Corpus size per runbook (raw, before cleaning)")
plt.gca().invert_yaxis()      # barh draws bottom-to-top by default; flip so the dict order reads top-to-bottom
plt.tight_layout(); plt.show()

def doc_card(name, text, width=60):
    """Print a plain-text 'card' — a poor-man's terminal UI, purely to make each
    document feel like a distinct object rather than a wall of text in a dict."""
    lines = [l.strip() for l in text.splitlines() if l.strip()][:6]   # first 6 non-blank lines
    print("+" + "-" * width + "+")
    print("| " + name.upper().ljust(width - 1) + "|")     # ljust pads with spaces to align the right border
    print("+" + "-" * width + "+")
    for l in lines:
        print("| " + l[:width - 2].ljust(width - 1) + "|")   # truncate long lines so they don't overflow the box
    print("+" + "-" * width + "+\n")

for n, t in RAW_RUNBOOKS.items():
    doc_card(n, t)

---
## Section 2 — Cleaning

**Why:** garbage in the corpus becomes garbage in the vectors. `CONFIDENTIAL - ACME Corp Internal`
appears in *every* document, so it carries **zero retrieval signal** — worse, it makes all documents
look slightly more similar to each other, which *blurs* search results.

**What we remove:**
- header/footer lines (`CONFIDENTIAL`, `Page N of M`)
- blank-line runs
- duplicate whitespace

**Production note:** in the real SRE Copilot, this is exactly what a Flink job does to raw logs before
they ever reach the LLM — filter noise *early*, at the cheapest point in the pipeline.


In [ ]:
# Each pattern below matches ONE kind of boilerplate line we saw in the raw text.
# `re.sub(pattern, "", text)` finds every match anywhere in the string and deletes it —
# not just at the start/end, so this also removes a header if it were repeated mid-document.
HEADER_FOOTER_PATTERNS = [
    r"CONFIDENTIAL.*",          # matches the literal word + everything after it on that line
    r"Page \d+ of \d+",         # \d+ = "one or more digits" -> matches "Page 1 of 2", "Page 12 of 34", etc.
]

def clean(text: str) -> str:
    for pat in HEADER_FOOTER_PATTERNS:
        text = re.sub(pat, "", text)                # strip each boilerplate pattern in turn
    text = re.sub(r"[ \t]+", " ", text)             # [ \t]+ = one-or-more spaces/tabs -> single space
                                                      # (deleting "CONFIDENTIAL" can leave double-spaces behind)
    text = re.sub(r"\n{3,}", "\n\n", text)          # \n{3,} = three-or-more consecutive newlines -> just two
                                                      # (removing a whole header line leaves an empty line gap)
    return text.strip()                              # trim leading/trailing whitespace from the whole doc

CLEAN_RUNBOOKS = {name: clean(t) for name, t in RAW_RUNBOOKS.items()}  # apply clean() to every runbook

# Before / after, side by side — just print the first 8 lines of one document from each dict
sample = "payment-service"
before = RAW_RUNBOOKS[sample].splitlines()[:8]
after  = CLEAN_RUNBOOKS[sample].splitlines()[:8]

print(f"{'BEFORE':45s} | AFTER")
print("-" * 95)
for b, a in zip(before, after + [""] * len(before)):   # pad `after` with "" so zip doesn't stop short
    print(f"{b[:44]:45s} | {a[:44]}")

# Sanity check: how many words did cleaning remove from EACH document? (word count before minus after)
removed = {n: len(RAW_RUNBOOKS[n].split()) - len(CLEAN_RUNBOOKS[n].split())
           for n in RAW_RUNBOOKS}
print("\nWords removed per document:", removed)

**Experiment:** comment out the `CONFIDENTIAL` pattern, re-run the whole notebook, and look at
Section 6's ranked search table. You'll see every document's similarity score rise slightly — for the
wrong reason. Noise inflates similarity uniformly.


---
## Section 3 — Chunking

**Why chunk at all?** Two reasons:
1. **Retrieval precision.** If we embed a whole runbook as one vector, a question about *restarting payments*
   matches the *entire* document — including its Redis and escalation sections. Smaller chunks = sharper matches.
2. **Prompt budget.** The LLM prompt has a token budget. We want to send only the 2–3 *most relevant* paragraphs,
   not five full runbooks.

The tension:
```
chunks too SMALL  →  precise matches, but context gets amputated (a command split in half)
chunks too LARGE  →  full context, but fuzzy matches and fat prompts
```
We'll make this tension *visible*.


In [ ]:
def chunk_words(text: str, chunk_size: int, overlap: int = 0):
    """Split text into chunks of `chunk_size` words, overlapping by `overlap` words."""
    assert overlap < chunk_size, "overlap must be smaller than chunk_size"
    words = text.split()                  # crude word split — good enough for this notebook, but
                                           # NOTE this is words, not tokens (a real tokenizer would
                                           # split "CrashLoopBackOff" into several sub-word pieces)
    step = chunk_size - overlap           # how far the window START advances each loop.
                                           # e.g. chunk_size=60, overlap=12 -> step=48, so the LAST
                                           # 12 words of chunk N are the SAME as the FIRST 12 of chunk N+1
    chunks = []
    for start in range(0, len(words), step):      # slide the window start by `step` words at a time
        piece = words[start : start + chunk_size]  # grab the next chunk_size words (Python slicing
                                                     # auto-truncates near the end, no IndexError risk)
        if len(piece) < 5:          # drop trailing crumbs — a 2-word "chunk" is mostly noise
            break
        chunks.append(" ".join(piece))
        if start + chunk_size >= len(words):   # this window already reached the end of the doc —
            break                               # stop, or the next slide would just repeat the tail
    return chunks

# How does chunk size change the shape of the corpus?
sizes = [50, 100, 300]
fig, ax = plt.subplots()
for size in sizes:
    counts = [len(chunk_words(t, size)) for t in CLEAN_RUNBOOKS.values()]
    ax.bar([f"{n}\n(size={size})" for n in CLEAN_RUNBOOKS], counts,
           label=f"chunk_size={size}", alpha=0.7)
ax.set_ylabel("number of chunks")
ax.set_title("Same corpus, three chunk sizes")
ax.legend()
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.tight_layout(); plt.show()

for size in sizes:
    total = sum(len(chunk_words(t, size)) for t in CLEAN_RUNBOOKS.values())
    print(f"chunk_size={size:4d} → {total:3d} total chunks")

### 3.1 The overlap problem — watching a command get amputated

Here is the failure mode that overlap exists to fix. Watch what happens to a `kubectl` command
at a chunk boundary when overlap = 0:


In [ ]:
demo_text = CLEAN_RUNBOOKS["payment-service"]

# Same source text, two different chunkers — chunk_size is fixed at 40 words so the only
# variable we're isolating is overlap (0 vs 10 words). This is the "change one thing" pattern.
no_overlap   = chunk_words(demo_text, chunk_size=40, overlap=0)
with_overlap = chunk_words(demo_text, chunk_size=40, overlap=10)

print("WITHOUT overlap — look for a shell command cut mid-flight:\n")
for i, c in enumerate(no_overlap):
    print(f"--- chunk {i} (last 12 words) ---")
    print("   ", " ".join(c.split()[-12:]))     # print just the TAIL of each chunk — that's where
                                                  # a command starting near the boundary would get cut
print()
print("WITH overlap=10 — the boundary words appear in BOTH chunks:\n")
for i, c in enumerate(with_overlap):
    print(f"--- chunk {i} (first 12 words) ---")
    print("   ", " ".join(c.split()[:12]))       # print just the HEAD of each chunk — compare this
                                                  # to the previous chunk's tail: they should overlap

**What to notice:** with `overlap=0`, a command like `kubectl rollout restart deployment payment-api`
can be split so that neither chunk contains the *complete* command — so *no* chunk is a good retrieval
target for "how do I restart payments?". With overlap, the boundary region lives in both chunks, so at
least one chunk always holds the intact command.

**Trade-off:** overlap duplicates data. `overlap=10` on `chunk_size=40` is 25% storage overhead.
In production (pgvector, Milestone 13) that's real disk and real embedding cost. Typical values: 10–20% of chunk size.

Now build the final chunk set we'll use for the rest of the notebook — and keep **metadata** with every chunk.
Metadata is what lets a real system answer "*which runbook did this come from?*"


In [ ]:
CHUNK_SIZE = 60
OVERLAP    = 12

corpus = []          # list of dicts — this IS our "document store" for the rest of the notebook
for doc_name, text in CLEAN_RUNBOOKS.items():
    # chunk_words() returns a LIST of chunks for one doc; enumerate() numbers them 0, 1, 2...
    # so each chunk gets a unique, traceable id like "kafka-cluster#0", "kafka-cluster#1"
    for i, chunk in enumerate(chunk_words(text, CHUNK_SIZE, OVERLAP)):
        corpus.append({
            "doc":      doc_name,           # which runbook this chunk came from (metadata)
            "chunk_id": f"{doc_name}#{i}",  # unique id used everywhere downstream to trace a result
            "text":     chunk,              # the actual chunk text that gets embedded
        })

df_corpus = pd.DataFrame(corpus)     # one row per chunk — this DataFrame stays aligned with the
                                      # TF-IDF matrix X we build next: row i of df_corpus == row i of X
print(f"{len(corpus)} chunks total")
df_corpus.head(8)

---
## Section 4 — Embeddings (TF-IDF: vectors you can read)

**The core idea of all retrieval:** turn text into numbers such that *similar meaning → nearby vectors*.

TF-IDF (Term Frequency × Inverse Document Frequency), in one breath:
- **TF** — a word that appears often in *this* chunk matters to this chunk.
- **IDF** — a word that appears in *every* chunk (`the`, `check`, `service`) matters to none of them.
- Score = TF × IDF → each chunk becomes a vector with **one dimension per vocabulary word**.

That last property is the whole reason we start here: you can point at any number in the vector
and say which *word* it belongs to. Neural embeddings (Notebook 2) trade this readability for the
ability to understand synonyms — "reboot" ≈ "restart" — which TF-IDF cannot do.


In [ ]:
# TfidfVectorizer builds the VOCABULARY (every distinct word across all chunks) and then
# scores each word in each chunk by TF (how often it appears in THIS chunk) x IDF (how RARE
# it is across ALL chunks). Common words score near zero; distinctive words score high.
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",                    # drop 'the', 'if', 'and', ... — near-zero IDF anyway,
                                              # since they appear in almost every chunk
    token_pattern=r"[a-zA-Z][a-zA-Z\-]+",     # regex controls what counts as a "word": letters plus
                                              # hyphens, so 'payment-api' stays ONE token instead of
                                              # splitting into 'payment' and 'api' (the default pattern would split it)
)

# fit_transform does two things at once:
#   1. fit  -> scans every chunk, builds the vocabulary (one column per distinct word)
#   2. transform -> converts each chunk's text into a row of TF-IDF weights over that vocabulary
# Result: a SPARSE matrix of shape (n_chunks, vocab_size) — sparse because most chunks
# use only a tiny fraction of the total vocabulary, so most entries are exactly 0.
X = vectorizer.fit_transform(df_corpus["text"])
vocab = np.array(vectorizer.get_feature_names_out())   # column i's weight corresponds to word vocab[i]

print(f"Embedding matrix shape: {X.shape}  → {X.shape[0]} chunks × {X.shape[1]} vocabulary dims")
print(f"Sparsity: {100 * (1 - X.nnz / (X.shape[0]*X.shape[1])):.1f}% of entries are zero")

### 4.1 Read one vector with your own eyes

Pick one chunk and look at its top-weighted dimensions. This is the "aha" cell.


In [ ]:
# Grab ROW INDEX (not chunk_id) of the first kafka-cluster chunk in df_corpus, so we can
# pull the matching row out of X (X and df_corpus are aligned — row i of X IS chunk i of df_corpus).
idx = df_corpus.index[df_corpus["doc"] == "kafka-cluster"][0]
row = X[idx].toarray().ravel()          # X is sparse; .toarray() makes this ONE row dense so we can
                                         # sort it. .ravel() flattens the (1, vocab_size) matrix to a
                                         # plain 1-D array of length vocab_size — one weight per word.
top = row.argsort()[::-1][:10]          # argsort() gives indices that would sort ascending;
                                         # [::-1] reverses to descending; [:10] keeps the top 10 weights.
                                         # top[i] is a COLUMN INDEX into row/vocab, not a weight itself.

print("CHUNK TEXT:")
print(textwrap.fill(df_corpus.loc[idx, "text"], 90))
print("\nTOP TF-IDF DIMENSIONS (word → weight):")
for t in top:                           # t is a vocabulary index; vocab[t] is the WORD, row[t] its weight
    bar = "█" * int(row[t] * 40)        # crude ASCII bar chart, scaled so weight~1.0 draws ~40 blocks
    print(f"  {vocab[t]:28s} {row[t]:.3f}  {bar}")

### 4.2 Heatmap: the whole corpus at a glance

Rows = chunks (grouped by runbook), columns = the 25 highest-signal words. Watch the block structure emerge:
Kafka words light up only on Kafka rows.


In [ ]:
dense = X.toarray()                              # convert the FULL sparse matrix to dense — fine at
                                                  # this tiny scale (10 chunks), would be wasteful at scale
# For each WORD (column), find its single highest weight across all chunks — a word that never
# scores highly anywhere is uninteresting to plot. Sort columns by that peak, descending, keep top 25.
top_terms = dense.max(axis=0).argsort()[::-1][:25]

fig, ax = plt.subplots(figsize=(12, 7))
im = ax.imshow(dense[:, top_terms], aspect="auto", cmap="viridis")   # slice OUT just those 25 columns
ax.set_xticks(range(len(top_terms)))
ax.set_xticklabels(vocab[top_terms], rotation=90, fontsize=8)        # label columns with the actual words
ax.set_yticks(range(len(df_corpus)))
ax.set_yticklabels(df_corpus["chunk_id"], fontsize=7)                # label rows with chunk ids
ax.set_title("TF-IDF weights — chunks × top-25 terms")
fig.colorbar(im, label="tf-idf weight")
plt.tight_layout(); plt.show()

---
## Section 5 — Visualizing the Embedding Space (PCA → 2D)

Our vectors live in a ~300-dimensional space. PCA squashes them to 2D while preserving as much
variance as possible — enough to *see* the geometry of retrieval:

**Prediction before you run it:** the three *service* runbooks (payment, order, login) share vocabulary
(kubectl, pods, restart, deployment) so they should cluster together, while Kafka should sit apart.


In [ ]:
# PCA finds the 2 directions (out of ~vocab_size dimensions) along which the chunks vary the
# MOST, and projects every vector onto just those 2 — a lossy compression chosen to preserve
# as much of the original spread-out-ness as possible, purely so we can plot it on a 2D screen.
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(dense)          # coords.shape == (n_chunks, 2) — one (x, y) point per chunk

colors = {"payment-service": "#4C72B0", "order-service": "#DD8452",
          "login-service": "#55A868", "kafka-cluster": "#C44E52",
          "kubernetes-deploys": "#8172B3"}

fig, ax = plt.subplots(figsize=(10, 7))
for doc_name, color in colors.items():
    mask = (df_corpus["doc"] == doc_name).values     # boolean array: True where this chunk belongs
                                                       # to doc_name — used to pull out just its rows
    ax.scatter(coords[mask, 0], coords[mask, 1], c=color, s=90,
               label=doc_name, edgecolors="black", linewidths=0.5)
ax.set_title(f"Chunk embedding space (PCA, {pca.explained_variance_ratio_.sum():.0%} variance kept)")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.legend()
plt.tight_layout(); plt.show()

**What to notice:**
- Chunks from the same runbook cluster — same vocabulary → similar vectors.
- Kafka sits apart from the services — different tooling vocabulary (`broker`, `partitions`, `lag`).
- kubernetes-deploys likely sits *near* the service runbooks — they all speak `kubectl`. This foreshadows
  a real retrieval hazard: **shared vocabulary creates false neighbors**. That's what re-ranking fixes in Notebook 2.

*(PCA keeps only a slice of the variance, so treat this as a map, not the territory.)*


---
## Section 6 — The Vector "Database" is a Python List

pgvector, FAISS, Pinecone — under the hood, a vector database does exactly two things:
1. **store** `(vector, metadata)` pairs
2. **return the k nearest vectors** to a query vector

So our first vector database is a list, and our search engine is a for-loop. This is not a toy —
it is the *actual algorithm* (exact brute-force k-NN). Real databases add indexing (HNSW, IVF)
only to make this loop fast at millions of vectors.


In [ ]:
# --- the "database" -------------------------------------------------
# A "vector database record" is just: the vector itself, plus enough metadata to
# turn a matching row-index back into something human-readable (which doc, which chunk).
VECTOR_DB = []
for i, rec in enumerate(corpus):                 # corpus[i] and dense[i] refer to the SAME chunk —
    VECTOR_DB.append({                            # they were built in the same loop, same order, so
        "vector":   dense[i],                     # index i is the join key between the two.
        "metadata": {"doc": rec["doc"], "chunk_id": rec["chunk_id"]},
        "text":     rec["text"],
    })
print(f"VECTOR_DB holds {len(VECTOR_DB)} records. Record 0 keys: {list(VECTOR_DB[0].keys())}")

# --- cosine similarity, by hand -------------------------------------
def cosine(a: np.ndarray, b: np.ndarray) -> float:
    """cos(θ) = (a·b) / (|a||b|)  — angle between vectors, ignores length."""
    # a @ b        = dot product: sum(a[i]*b[i] for all i) — bigger when vectors point the same way
    # |a| * |b|    = product of vector LENGTHS (np.linalg.norm = sqrt(sum of squares))
    # dividing by the lengths is what makes this "cosine" rather than raw dot product:
    # it normalizes out magnitude, so a SHORT chunk and a LONG chunk about the same topic
    # can still score close to 1.0 even though their raw dot product would differ a lot.
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(a @ b / denom) if denom else 0.0    # guard against divide-by-zero for an all-zero vector

# sanity checks — build intuition for the metric itself before trusting it on real queries
v_kafka1 = dense[df_corpus.index[df_corpus.doc == "kafka-cluster"][0]]
v_kafka2 = dense[df_corpus.index[df_corpus.doc == "kafka-cluster"][1]]
v_login  = dense[df_corpus.index[df_corpus.doc == "login-service"][0]]

print(f"\ncos(kafka chunk, kafka chunk) = {cosine(v_kafka1, v_kafka2):.3f}   ← same topic, high")
print(f"cos(kafka chunk, login chunk) = {cosine(v_kafka1, v_login):.3f}   ← different topic, low")

---
## Section 7 — Similarity Search (the most important cell in the notebook)

The full retrieval algorithm:
```
question → same vectorizer → query vector
for every chunk: cosine(query, chunk)
sort descending → take top-k
```
One critical rule: the query **must** be embedded by the **same vectorizer** that embedded the corpus.
A query vector from a different vocabulary/model lives in a different space — similarities become meaningless.
(This exact bug happens in production when someone upgrades the embedding model without re-indexing.)


In [ ]:
def search(question: str, k: int = 3) -> pd.DataFrame:
    # .transform() (NOT .fit_transform()) — reuses the vocabulary already learned from the corpus.
    # A word in the question that never appeared in the corpus is simply IGNORED (no new column
    # gets created), which is exactly the "same vector space" requirement stated above.
    q_vec = vectorizer.transform([question]).toarray().ravel()
    scored = [                                         # brute-force: compare the query against EVERY
        {"score": cosine(q_vec, rec["vector"]),         # record in the "database" — O(n) per search,
         "chunk_id": rec["metadata"]["chunk_id"],       # fine at 10 chunks, the reason real systems
         "doc": rec["metadata"]["doc"],                 # need an index (FAISS in Notebook 2, HNSW in
         "text": rec["text"]}                           # Notebook 3) once n gets into the thousands+
        for rec in VECTOR_DB
    ]
    return (pd.DataFrame(scored)
              .sort_values("score", ascending=False)    # highest similarity first
              .head(k)                                  # keep only the top k results
              .reset_index(drop=True))

question = "How do I restart the payment service?"
results = search(question, k=5)
results[["score", "chunk_id", "text"]].style.bar(subset=["score"], color="#4C72B0")

In [ ]:
# Visualize the ranking as a bar chart — same `results` DataFrame computed above, just plotted
fig, ax = plt.subplots()
ax.barh(results["chunk_id"], results["score"], color="#4C72B0")
ax.invert_yaxis()          # barh plots bottom-to-top by default; invert so highest score is at the top
ax.set_xlabel("cosine similarity")
ax.set_title(f'Top-5 chunks for: "{question}"')
plt.tight_layout(); plt.show()

### 7.1 Probe the boundaries of TF-IDF

Run these and study the failures as carefully as the successes:


In [ ]:
for q in [
    "Why is Kafka consumer lag increasing?",          # should nail kafka-cluster
    "Users cannot log in, seeing 401 errors",         # should nail login-service
    "The message bus is falling behind",              # ← TF-IDF BLINDNESS: means Kafka lag, scores ~0.0
    "Customers see failures at checkout",             # ← COLLISION: 'failures' pulls in the k8s runbook
    "kubectl rollout",                                 # ← shared vocab: many docs match
]:
    top = search(q, k=3)
    print(f"Q: {q}")
    for _, r in top.iterrows():
        print(f"   {r.score:.3f}  {r.chunk_id}")
    print()

**Two failures worth memorizing:**
1. **Synonym blindness** — "The message bus is falling behind" *means* "Kafka consumer lag" to any SRE,
   yet it scores ≈ 0.000 because it shares **zero tokens** with the Kafka runbook. TF-IDF matches
   *strings*, not *meaning*. Neural embeddings (Notebook 2) fix exactly this.
2. **Vocabulary collision** — "Customers see failures at checkout" retrieves the **Kubernetes** runbook,
   not payments, because the word "failures/failing" is strong in the k8s doc. Retrieval matched the
   right *word* in the wrong *context*. That's the job of re-ranking with a cross-encoder (Notebook 2).


---
## Section 8 — Prompt Building (see the actual prompt)

RAG's final trick is almost embarrassingly simple: **paste the retrieved chunks into the prompt**.
Most people never look at the assembled prompt. You will — because prompt inspection is the #1
debugging tool for RAG systems ("did the model hallucinate, or did retrieval feed it the wrong context?").

Design decisions visible below (each one is a real production choice):
- chunks are **labeled with their source** → enables citations
- instructions demand **grounding** ("answer ONLY from context") → reduces hallucination
- an explicit **escape hatch** ("say you don't know") → the model needs permission to refuse


In [ ]:
def build_prompt(question: str, k: int = 3) -> str:
    retrieved = search(question, k)                      # re-run the search() from Section 7
    context_blocks = "\n\n".join(                        # stitch the k retrieved chunks into one
        f"[Source: {r.chunk_id} | similarity={r.score:.2f}]\n{r.text}"   # string, each one tagged with
        for _, r in retrieved.iterrows()                  # its chunk_id and score so the model (and you,
    )                                                      # reading the printed prompt) can trace which
                                                            # chunk any given claim came from
    return f"""You are an SRE assistant. Answer the on-call engineer's question using ONLY the context below.
Cite the source chunk id for every instruction you give.
If the context does not contain the answer, say "I don't have a runbook for that" — do not guess.

=== CONTEXT ===
{context_blocks}

=== QUESTION ===
{question}

=== ANSWER ==="""

prompt = build_prompt("How do I restart the payment service?", k=3)
print(prompt)
# rough token estimate: OpenAI-family tokenizers average ~4 characters per token for English text
print(f"\n--- prompt stats: {len(prompt)} chars ≈ {len(prompt)//4} tokens ---")

**Experiment:** change `k=3` to `k=8` and watch the token estimate. Every retrieved chunk costs
tokens → latency → money. In Milestone 39 of the main project, this exact knob is a cost-optimization lever.


---
## Section 9 — The LLM Call (optional; mock included)

The pipeline is complete without an API key: the cell below includes a **mock LLM** that extracts an
answer mechanically, so end-to-end runs are free. When you're ready, paste a key into the real call.


In [ ]:
def mock_llm(prompt: str) -> str:
    """Fake 'LLM': proves the pipeline works end-to-end with zero API cost.

    Note a subtle bug we hit while writing this: our word-based chunker did
    " ".join(words), which DESTROYED newlines — so extracting 'command lines'
    line-by-line found nothing. Real lesson: chunking can silently discard
    document structure (line breaks, tables, code blocks). Structure-aware
    chunking exists for exactly this reason. Here we recover commands by regex.
    """
    # pull just the CONTEXT section out of the full prompt string (between the two markers)
    context = prompt.split("=== CONTEXT ===")[1].split("=== QUESTION ===")[0]
    # match a command word (kubectl/redis-cli/systemctl/kafka-*) followed by its arguments.
    # (?:...)     = non-capturing group, just for grouping the alternation
    # \s+(?!...)  = "one-or-more spaces, but NOT if the next char is uppercase" — this stops the
    #               match before it swallows the next sentence, which usually starts capitalized
    cmd_pattern = r"(?:kubectl|redis-cli|systemctl|kafka-[\w.\-]+)(?:\s+(?![A-Z])[\w./<>#=\-]+)*"
    commands = re.findall(cmd_pattern, context)      # every match anywhere in the context, as a list
    if not commands:
        return "I don't have a runbook for that."
    # dict.fromkeys(commands) de-duplicates while PRESERVING first-seen order (a plain set() wouldn't)
    steps = "\n".join(f"{i+1}. `{c}`" for i, c in enumerate(dict.fromkeys(commands)))
    return f"Based on the retrieved runbook chunks, run:\n{steps}"

print(mock_llm(build_prompt("How do I restart the payment service?")))

In [ ]:
# ---- OPTIONAL: real LLM call (uncomment + add a key) ----------------
# In Colab: add your key via the 🔑 "Secrets" panel, then:
#
# from google.colab import userdata
# from openai import OpenAI
# client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
#
# def real_llm(prompt: str) -> str:
#     resp = client.chat.completions.create(
#         model="gpt-4o-mini",
#         messages=[{"role": "user", "content": prompt}],
#         temperature=0,           # deterministic answers for runbooks
#     )
#     return resp.choices[0].message.content
#
# print(real_llm(build_prompt("How do I restart the payment service?")))
print("Skipping real LLM call — mock is active.")

---
## Section 10 — Experiments: turn the knobs, watch retrieval change

This section is the point of the whole notebook. We rebuild the index at several chunk sizes and
measure how well retrieval finds the *right document* for a set of test questions. This is your first
**retrieval evaluation** — the primitive version of what RAGAS does in Notebook 3.


In [ ]:
TEST_SET = [
    # -------- easy: vocabulary matches directly --------
    ("How do I restart the payment service?",       "payment-service"),
    ("Why is Kafka consumer lag increasing?",       "kafka-cluster"),
    ("Users are getting 401 errors on login",       "login-service"),
    ("Orders are stuck in pending state",           "order-service"),
    ("How do I fix under-replicated partitions?",   "kafka-cluster"),
    # -------- hard: verified failures of TF-IDF (run them and see!) --------
    ("The message bus is falling behind",           "kafka-cluster"),     # 0 shared words → score 0.0
    ("Sign-in is broken for everyone",              "login-service"),     # 'sign-in' ≠ 'login' as tokens
    ("Customers see failures at checkout",          "payment-service"),   # 'failures' collides with k8s runbook!
    ("The event pipeline stopped delivering data",  "kafka-cluster"),     # pure paraphrase
    ("Purchases are timing out",                    "order-service"),     # 'timing out' collides with login runbook
]

def build_index(chunk_size, overlap):
    """Re-chunk the ENTIRE corpus at a given (chunk_size, overlap) and fit a FRESH
    TfidfVectorizer on it. A fresh vectorizer matters — different chunking means different
    chunks means a potentially different vocabulary, so we can't reuse the Section 4 vectorizer."""
    recs = []
    for doc_name, text in CLEAN_RUNBOOKS.items():
        for i, ch in enumerate(chunk_words(text, chunk_size, overlap)):
            recs.append({"doc": doc_name, "text": ch})
    vec = TfidfVectorizer(lowercase=True, stop_words="english",
                          token_pattern=r"[a-zA-Z][a-zA-Z\-]+")
    mat = vec.fit_transform([r["text"] for r in recs]).toarray()   # (n_chunks_at_this_size, vocab_size)
    return recs, vec, mat

def hit_at_1(chunk_size, overlap):
    """Build an index at this (chunk_size, overlap), then check: for each test question,
    does the SINGLE best-scoring chunk belong to the expected document? Returns the fraction
    of questions that got it right — this is the vectorized version of cosine() from Section 6,
    computed against every chunk at once instead of one pair at a time."""
    recs, vec, mat = build_index(chunk_size, overlap)
    hits = 0
    for q, expected_doc in TEST_SET:
        qv = vec.transform([q]).toarray().ravel()          # embed the question in THIS index's vocabulary
        # mat @ qv               -> dot product of the query against EVERY row of mat at once (n_chunks,)
        # np.linalg.norm(mat, axis=1) -> the LENGTH of each chunk's vector (one norm per row)
        # dividing element-wise gives cosine similarity for all chunks in a single vectorized expression
        # — this is exactly what the earlier cosine() function does, just for every chunk simultaneously
        # instead of a Python for-loop. +1e-9 avoids a divide-by-zero for an all-empty query vector.
        sims = mat @ qv / (np.linalg.norm(mat, axis=1) * np.linalg.norm(qv) + 1e-9)
        best = recs[int(sims.argmax())]        # argmax -> row index of the single highest score
        hits += (best["doc"] == expected_doc)  # True/False adds as 1/0 to the running count
    return hits / len(TEST_SET)

# Sweep every combination of chunk_size and overlap-as-a-fraction-of-chunk_size,
# rebuilding the WHOLE index and re-scoring the WHOLE test set at each setting.
rows = []
for cs in [20, 40, 60, 100, 200]:
    for ov_frac in [0.0, 0.2]:
        ov = int(cs * ov_frac)         # e.g. cs=100, ov_frac=0.2 -> overlap=20 words
        rows.append({"chunk_size": cs, "overlap": ov,
                     "hit@1": hit_at_1(cs, ov)})
df_eval = pd.DataFrame(rows)

pivot = df_eval.pivot(index="chunk_size", columns="overlap", values="hit@1")   # (unused below, but
                                                                                 # handy to inspect yourself)
print(df_eval.to_string(index=False))

# Split the results back into the two overlap settings so we can plot them as separate lines
df_eval_plot = df_eval[df_eval.overlap == 0]
plt.plot(df_eval_plot.chunk_size, df_eval_plot["hit@1"], "o-", label="overlap=0")
df_eval_plot2 = df_eval[df_eval.overlap > 0]
plt.plot(df_eval_plot2.chunk_size, df_eval_plot2["hit@1"], "s--", label="overlap=20%")
plt.xlabel("chunk size (words)"); plt.ylabel("hit@1 on test set")
plt.title("Retrieval quality vs chunk size")
plt.ylim(0, 1.05); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**Read the flat line — it's the most important result in this notebook.**

hit@1 sits at ~0.5 for *every* chunk size and overlap. The 5 lexical questions always hit; the 5
paraphrased questions always miss. Turning the chunking knobs changes *nothing*, because the failure
isn't in chunking — it's that **TF-IDF cannot represent meaning, only vocabulary**.

This is a pattern you will meet constantly in production RAG: teams burn weeks tuning chunk size
when their real bottleneck is the embedding model (or vice versa). An evaluation set that separates
*lexical* from *semantic* questions — like ours — is how you find out **which knob actually matters**
before you start turning them. Notebook 2 swaps in a sentence-transformer and re-runs this exact
table; predict now what happens to each half of the test set.

**Try next (self-directed):**
1. Add 4 more test questions, including one *not answerable* by any runbook. What should hit@1 mean then?
2. Set `overlap = 0` and `chunk_size = 15` — find a question that fails *only* because a command was split
   (hint: hit@1 checks the *document*, so you'll need to check the *chunk* to see this failure).
3. Change `k` in `build_prompt` from 1 → 10 and eyeball prompt token counts. Where would you set the budget?
4. Remove `stop_words="english"` from the vectorizer. Which questions get worse, and why?


---
## Conclusion — what you actually built

```
text → clean → chunk(+overlap) → vectorize → store(list) → cosine top-k → prompt → LLM
```

Every production RAG system — including the SRE Copilot's Milestone 13 (pgvector) and
Milestones 16–19 (LangGraph retrieval tools) — is this same pipeline with better parts:

| This notebook          | Notebook 2            | Production (SRE Copilot)     |
|------------------------|-----------------------|------------------------------|
| TF-IDF                 | Sentence Transformers | text-embedding-3 / BGE       |
| Python list            | FAISS                 | PostgreSQL + pgvector        |
| for-loop cosine        | ANN index (HNSW)      | pgvector HNSW index          |
| top-k only             | + cross-encoder rerank| + hybrid BM25 + rerank       |
| manual prompt          | template              | LangGraph retrieval tool     |

### Quiz (answer before opening Notebook 2)

1. Why does removing the `CONFIDENTIAL` header improve retrieval, even though cosine similarity is normalized?
2. A teammate embeds queries with model A but the corpus was indexed with model B. Search "works" (no errors) but results are garbage. Explain why, and name the production event that commonly causes this bug.
3. You saw "reboot the payments component" fail. Explain *mechanically* why TF-IDF cannot fix this, no matter how you tune chunk size.
4. Chunk overlap costs storage and embedding compute. Give one scenario where you'd set overlap to 0 on purpose.
5. In `build_prompt`, why do we include the *chunk id* in the context, and what production feature does it enable?
